In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Object Counting in Video Using YOLO

## Objective

The objective is to detect, track, and count moving objects
crossing a defined line in a video.

This task simulates traffic monitoring and vehicle counting systems.

In [ ]:
!pip install ultralytics

In [8]:
import cv2
from ultralytics import YOLO
from ultralytics.solutions import ObjectCounter
from google.colab import files

In [ ]:
model = YOLO("yolov8n.pt")
for class_id, class_name in model.names.items():
    print(class_id, class_name)

## Pretrained YOLO Model

A pretrained model is a model that has already learned
from a large dataset.

We use a COCO pretrained model because it can detect
common objects such as people, cars, buses, trucks,
motorcycles, and bicycles.

Class names represent the categories that the model can detect.

In [ ]:
uploaded = files.upload()

video_path = list(uploaded.keys())[0]

print("Video:", video_path)

In [ ]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("Could not open the video!")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Width:", width)
print("Height:", height)
print("FPS:", fps)
print("Total Frames:", total_frames)

cap.release()

## Video Information

FPS means Frames Per Second. It describes how many frames
are displayed in one second.

Width and height represent the video dimensions.
They are required when creating the output video.

Checking whether the video opened successfully prevents
processing errors.

In [ ]:
region_points = [
    (int(width * 0.1), int(height * 0.5)),
    (int(width * 0.9), int(height * 0.5))
]

print("Counting line points:", region_points)

In [ ]:
counter = solutions.ObjectCounter(
    model="yolo11n.pt",
    region=region_points,
    classes=[0, 2, 3, 5, 7],
    show=False,
    verbose=False
)

print("Object Counter initialized successfully!")

In [ ]:
cap = cv2.VideoCapture(video_path)

output_path = "object_counting_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

video_writer = cv2.VideoWriter(
    output_path,
    fourcc,
    fps if fps > 0 else 25,
    (width, height)
)

if not video_writer.isOpened():
    raise Exception("Could not create output video!")

frame_count = 0

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    results = counter(frame)

    annotated_frame = results.plot_im

    video_writer.write(annotated_frame)

    frame_count += 1

    if frame_count % 100 == 0:
        print("Processed frames:", frame_count)

cap.release()
video_writer.release()

print("Video processing completed!")

In [ ]:
print("Objects moving IN:", counter.in_count)
print("Objects moving OUT:", counter.out_count)

print("Class-wise Count:")

for class_name, counts in counter.classwise_count.items():
    print(class_name, counts)

In [ ]:
from IPython.display import Video, display

display(Video(output_path, embed=True))

In [ ]:
files.download(output_path)

## Object Counting Explanation

### Detection vs Tracking

Detection identifies objects in each frame.

Tracking follows the same objects across multiple frames
and assigns IDs to them.

### Why Tracking is Important

Tracking helps the system understand object movement
and avoid counting the same object multiple times.

### Object IDs

Each tracked object has a unique ID.

The counter uses object positions and tracking information
to determine whether an object has crossed the counting line.

### Final Observations

The system detects and tracks objects and counts
their movement across the selected line.

Counting accuracy depends on camera position,
video quality, object visibility, and tracking performance.